# Using SynHydro to Compare the Valencia-Schaake and Nowak Temporal Disaggregation Methods

Earlier this summer I made [a post introducing SynHydro](https://waterprogramming.wpcomstaging.com/2026/05/28/introducing-synhydro-a-python-package-providing-access-to-many-synthetic-streamflow-generation-methods/) (`synhydro`), a Python package providing access to many synthetic streamflow generation methods through a unified API. That post was a broad overview of the package which didn't get into the method details or deeper comparisons.

In this follow-up post, I focus on the details of the streamflow disaggregation tools currently available in the SynHydro package.

The focus here is *temporal disaggregation*. Many of the stochastic streamflow generators across the literature and in `synhydro` operate at annual or monthly timesteps, where distributional assumptions are more defensible and there are fewer parameters to estimate. Water resources applications usually need finer resolution than that. Disaggregation methods make this possible. Given synthetic coarse-timestep flows, they produce realistic finer-timestep sequences that sum back to the coarse totals.

Disaggregation is nearly as old as synthetic hydrology generation methods themselves. Valencia and Schaake (1973) formalized the parametric linear approach only a decade after the Thomas-Fiering model ([which I wrote about previously](https://waterprogramming.wpcomstaging.com/2024/02/08/the-thomas-fiering-model-for-synthetic-streamflow-generation-with-a-python-implementation/)), and it remains the classical baseline of the disaggregation literature. Almost four decades later, Nowak et al. (2010) approached the same problem nonparametrically with K-nearest-neighbor (KNN) resampling. Longtime blog readers may already be familiar with the Nowak method as the monthly-to-daily step of the Kirsch-Nowak generation, covered in Julie Quinn's [Open Source Streamflow Generator posts](https://waterprogramming.wpcomstaging.com/2017/08/29/open-source-streamflow-generator-part-i-synthetic-generation/). The Valencia-Schaake method, despite its foundational status, has not made an appearance on this blog until now.

SynHydro implements both methods behind the same `fit`/`disaggregate` interface, which makes it easy to run them side by side.

In this post I generate a single annual ensemble at four Delaware River Basin gauges, disaggregate it to monthly flows with each method, and compare the results as we go.

## The Valencia-Schaake method (1973)

The Valencia-Schaake model (Valencia and Schaake, 1973) treats the vector of sub-period flows as a linear function of the aggregate flows, plus Gaussian noise. Following the notation of the original paper, let $m$ be the number of sites and $s$ the number of sub-periods per year (here $s = 12$ months). All monthly flows for one year are stacked in the vector $\mathbf{Y} \in \mathbb{R}^{sm}$ (all 12 months for site 1, then site 2, and so on), and $\mathbf{X} \in \mathbb{R}^{m}$ holds the per-site annual totals. The model is

$$
\mathbf{Y} = \boldsymbol{\mu}_Y + \mathbf{A}\,(\mathbf{X} - \boldsymbol{\mu}_X) + \mathbf{B}\,\mathbf{V}, \qquad \mathbf{V} \sim \mathcal{N}(\mathbf{0},\, \mathbf{I})
$$

where the parameter matrices $\mathbf{A}$ and $\mathbf{B}$ are estimated from sample covariance and cross-covariance matrices $\mathbf{S}$ of the historical record (paper Eqs. 14, 15, and 19)

$$
\mathbf{A} = \mathbf{S}_{yx}\,\mathbf{S}_{xx}^{-1}, \qquad \mathbf{B}\mathbf{B}^\top = \mathbf{S}_{yy} - \mathbf{S}_{yx}\,\mathbf{S}_{xx}^{-1}\,\mathbf{S}_{xy}.
$$

Because the model carries the full joint covariance of all $sm$ month-site combinations, cross-site and cross-month correlations are preserved by design.

The most elegant result in the paper (Eqs. 31-42) concerns mass conservation. Define the aggregation operator $\mathbf{C}$ such that $\mathbf{X} = \mathbf{C}\mathbf{Y}$ (each row of $\mathbf{C}$ sums one site's months). When $\mathbf{A}$ and $\mathbf{B}$ are estimated from historical data satisfying that identity, they inherit the properties

$$
\mathbf{C}\mathbf{A} = \mathbf{I}, \qquad \mathbf{C}\mathbf{B} = \mathbf{0},
$$

so every draw satisfies $\mathbf{C}\mathbf{Y} = \mathbf{X}$ exactly. The twelve monthly flows sum to the annual total at every site, for every realization, with no post-hoc correction.

There is one practical caveat. Monthly streamflow is strongly skewed, and the paper notes that transforming the data toward normality is "convenient but not absolutely necessary." SynHydro's implementation defaults to fitting in log space, which handles the skew but breaks the exact linear additivity above. Per-site conservation is then restored by proportionally rescaling each site's months to the annual total.

## The Nowak method (2010)

Nowak et al. (2010) take a nonparametric view of the same problem. Rather than assume a distributional form for the within-year pattern, they resample the patterns that actually occurred. The original paper presents the method at annual-to-daily resolution but notes that it "can be readily applied to any space and time scales". Here I use it at annual-to-monthly so that both methods solve the identical problem.

For each synthetic annual total $Q_p^{\text{syn}}$, the method finds the $K$ nearest historical years by annual flow magnitude at an index gauge (for multisite data, the sum across sites). One donor year is then sampled from the $K$ neighbors, with selection probabilities given by the kernel of Lall and Sharma (1996)

$$
w_i = \frac{1/i}{\displaystyle\sum_{j=1}^{K} 1/j}, \qquad i = 1, \ldots, K
$$

so the closest analog is roughly twice as likely to be chosen as the second closest, regardless of the absolute distances. The donor year's observed monthly flows $q_t^*$ define a proportion vector that is rescaled to the synthetic annual total

$$
q_t^{\text{syn}} = Q_p^{\text{syn}} \cdot \frac{q_t^*}{\displaystyle\sum_{t'=1}^{T} q_{t'}^*}.
$$

Annual totals are preserved by construction, and every synthetic within-year pattern is a rescaled historical one. For multisite data the same donor year is applied at every site, which preserves spatial consistency without estimating any covariance matrices.

SynHydro's default neighbor weighting is inverse-distance. In this demo I pass `sample_method="lall_and_sharma_1996"` so that the selection kernel matches the published method.

## The two methods at a glance

| | Valencia-Schaake | Nowak KNN |
|---|---|---|
| Type | Parametric (linear model, Gaussian noise) | Nonparametric (KNN resampling) |
| Input to output timescales | Annual to {12, 6, 4, 3, or 2} sub-periods | {Annual, monthly, weekly} to {monthly, weekly, daily} |
| Multisite | Yes, through the joint covariance of all site-month pairs | Yes, through a shared donor year across sites |
| Conserves input totals | Exactly | Exactly |
| Produces patterns not in the record | Yes | No, recombines observed patterns |
| Key requirement | Record long enough to estimate the joint covariance | Record long enough to span the relevant patterns |

Both methods are stochastic, meaning that disaggregating the same annual trace twice will give two different monthly sequences.

These two methods approach the same problem from opposite directions, which makes them a valuable pair to compare. Running them on the same input shows how much of the final ensemble behavior is inherited from the disaggregation step, and the rest of this post does exactly that.

## Setup and data

SynHydro ships with an example dataset containing daily streamflow (cms) for four USGS gauges in the Delaware River Basin, beginning in 1945. I trim to complete calendar years (the record ends partway through 2025) and aggregate to monthly and annual totals. This notebook lives at [`examples/blog_post/`](https://github.com/TrevorJA/SynHydro) in the SynHydro repository.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import synhydro
from synhydro.plotting import plot_seasonal_cycle, plot_monthly_distributions

FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

Q_daily = synhydro.load_example_data()
Q_daily = Q_daily.loc["1945":"2024"]  # keep complete calendar years only
Q_monthly = Q_daily.resample("MS").sum()
Q_annual = Q_daily.resample("YS").sum()

print(f"Sites: {list(Q_daily.columns)}")
print(f"{Q_annual.shape[0]} years, {Q_monthly.shape[0]} months")

## A shared annual input ensemble

To make this a controlled comparison, both disaggregators receive the *same* annual ensemble, so any differences in the monthly output are attributable to the disaggregation step alone.

For the annual generator I use SynHydro's `KNNBootstrapGenerator`. In plain terms, this generator builds each new synthetic year by finding the historical years whose flows most resemble the previously generated year, then sampling one of those years' annual flow vectors with a preference for the closest matches. It uses the same Lall and Sharma (1996) kernel that appears inside the Nowak method. Fifty realizations of fifty years each is plenty for stable statistics and runs in seconds.

In [ ]:
gen = synhydro.KNNBootstrapGenerator()
gen.fit(Q_annual)
annual_ens = gen.generate(n_realizations=50, n_years=50, seed=42)

The `generate()` call returns an `Ensemble` object, which is the standard data container in SynHydro. The `annual_ens` object holds all 50 synthetic annual traces along with metadata such as the time resolution and site names. The disaggregator classes always expect an `Ensemble` as input, and they return a new `Ensemble` at the finer timestep. The [ensembles tutorial](https://trevorja.github.io/SynHydro/tutorials/04_ensembles/) covers this object in more detail.

## Disaggregating with each method

Both disaggregators are fit on the observed *monthly* record. The Nowak method builds its donor pool from the observed within-year patterns, and Valencia-Schaake estimates its covariance matrices from the monthly vectors, computing the corresponding annual totals internally. Disaggregation is then a single call on the annual ensemble.

In [ ]:
vs = synhydro.ValenciaSchaakeDisaggregator(n_subperiods=12, transform="log")
vs.fit(Q_monthly)
vs_ens = vs.disaggregate(annual_ens, seed=42)

nowak = synhydro.NowakDisaggregator(input_timestep="annual", output_timestep="monthly")
nowak.fit(Q_monthly)
nowak_ens = nowak.disaggregate(annual_ens, sample_method="lall_and_sharma_1996", seed=42)

print(f"Monthly output: {vs_ens.data_by_realization[0].shape} per realization")

Both `vs_ens` and `nowak_ens` are new `Ensemble` objects containing the monthly flows. Each one holds 50 realizations of 600 monthly values at all four sites, and each realization corresponds to one annual trace from `annual_ens`.

## Results

Both disaggregators received the same annual ensemble, so we can pick a single realization and year and see what each method did with an identical annual total. Below are the driest and wettest input years from the first realization at the Port Jervis gauge, with one monthly trace from each method.

In [ ]:
site = Q_monthly.columns[0]
r = annual_ens.realization_ids[0]
annual_r = annual_ens.data_by_realization[r][site]
dry_year = annual_r.idxmin().year
wet_year = annual_r.idxmax().year

months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, year, label in zip(axes, [dry_year, wet_year],
                           ["Driest input year", "Wettest input year"]):
    vs_trace = vs_ens.data_by_realization[r].loc[str(year), site]
    nowak_trace = nowak_ens.data_by_realization[r].loc[str(year), site]
    total = annual_r[annual_r.index.year == year].iloc[0]
    ax.plot(range(1, 13), vs_trace.values, "-o", label="Valencia-Schaake")
    ax.plot(range(1, 13), nowak_trace.values, "-s", label="Nowak KNN")
    ax.set_xticks(range(1, 13), months, rotation=45)
    ax.set_title(f"{label} (annual total = {total:,.0f} cms)")
    ax.set_ylabel("Streamflow (cms)")
    ax.grid(alpha=0.3)
    ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "fig1_single_year_traces.png", dpi=150, bbox_inches="tight")

There are a few things worth noticing in these traces.

First, the two traces within each panel sum to exactly the same annual total, so any differences between them come entirely from the within-year allocation.

Second, the Nowak trace is a rescaled copy of a real historical year. Because of this, it looks like a typical Delaware River Basin hydrograph, with a spring melt peak, a summer recession, and fall rewetting.

Third, the Valencia-Schaake trace is a fresh draw from a fitted conditional distribution. It follows the same seasonal pattern in expectation, but each month is sampled around that pattern, and individual traces can wander in ways that no observed year does.

A single trace does not tell us much about the statistical behavior of each method, so the next comparison looks at the full ensemble. The panels below show the seasonal cycle of the monthly mean (top) and monthly standard deviation (bottom), computed separately for each realization. The observed statistic is shown as the dark line, and the band shows the 10th-90th percentile range across the 50 realizations.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharey="row")
for j, (ens, name) in enumerate([(vs_ens, "Valencia-Schaake"), (nowak_ens, "Nowak KNN")]):
    plot_seasonal_cycle(ens, observed=Q_monthly[site], site=site,
                        statistic="mean", ax=axes[0, j],
                        title=f"{name}: monthly mean")
    plot_seasonal_cycle(ens, observed=Q_monthly[site], site=site,
                        statistic="std", ax=axes[1, j],
                        title=f"{name}: monthly std")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2_seasonal_cycle.png", dpi=150, bbox_inches="tight")

Both methods reproduce the observed mean seasonal cycle well. For Valencia-Schaake this follows from the conditional mean being linear in the annual total, and for Nowak it follows from the resampling of observed monthly proportions.

The standard deviation panels reveal a clearer difference between the two methods. Nowak tracks the observed monthly standard deviation in every month, while Valencia-Schaake runs low from September through November, and the observed September value sits at the edge of the ensemble band.

This difference is explained by the shape of the fall flow distributions. The fall months in this basin mix low baseflows with occasional storm-driven floods, so their distributions are strongly skewed. Valencia-Schaake fits variability in log space, and the lognormal shape this implies underestimates the real-space variance of the most skewed months.

The month-by-month flow distributions shown below (log scale) give the clearest picture of the parametric and nonparametric behavior.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharey=True)
plot_monthly_distributions(vs_ens, observed=Q_monthly[site], site=site,
                           ax=axes[0], log_scale=True,
                           title="Valencia-Schaake")
plot_monthly_distributions(nowak_ens, observed=Q_monthly[site], site=site,
                           ax=axes[1], log_scale=True,
                           title="Nowak KNN")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig3_monthly_distributions.png", dpi=150, bbox_inches="tight")

The Nowak boxes sit nearly on top of the observed boxes, which is expected since resampling observed patterns reproduces the observed distributions more or less by definition. Valencia-Schaake extends past the observed range in both directions, producing both drier and wetter months than anything in the record.

Whether that extrapolation is a feature or a problem depends on the application. If you are stress-testing a system beyond the historical record, then generating unobserved conditions is the point. If your results hinge on low flows, keep in mind that those synthetic extremes come from a lognormal tail rather than from any observed event.

The next check looks at mass conservation. Both methods promise that the monthly flows sum back to exactly the annual totals we fed in. Valencia-Schaake accomplishes this through the $\mathbf{C}\mathbf{B} = \mathbf{0}$ construction along with the proportional rescale under the log transform, while Nowak accomplishes it through its renormalized proportion vectors. The check below re-aggregates each method's monthly output and compares against the input ensemble.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)
for ax, (ens, name) in zip(axes, [(vs_ens, "Valencia-Schaake"), (nowak_ens, "Nowak KNN")]):
    max_rel_err = 0.0
    for r_id in ens.realization_ids:
        annual_out = ens.data_by_realization[r_id].resample("YS").sum()
        annual_in = annual_ens.data_by_realization[r_id]
        rel_err = np.abs(annual_out.values - annual_in.values) / annual_in.values
        max_rel_err = max(max_rel_err, rel_err.max())
        if r_id < 10:  # scatter a subset to keep the figure light
            ax.scatter(annual_in.values.ravel(), annual_out.values.ravel(),
                       s=8, alpha=0.3, color="cornflowerblue")
    lims = ax.get_xlim()
    ax.plot(lims, lims, "k--", lw=1, label="1:1 line")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Input annual total (cms)")
    ax.set_title(f"{name}\nmax relative error = {max_rel_err:.1e}")
    ax.legend()
    print(f"{name}: max relative closure error = {max_rel_err:.2e}")
axes[0].set_ylabel("Re-aggregated annual total (cms)")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig4_annual_closure.png", dpi=150, bbox_inches="tight")

Both methods close the annual totals to floating point precision. This property matters because it means disaggregation does not disturb whatever statistics your annual generator was validated on. The annual-scale behavior is inherited exactly, and the disaggregator only controls how each year's total is distributed among the months.

The last comparison uses SynHydro's `verify()` function, which scores an ensemble against the observed record across suites of statistical metrics. Rather than print the resulting table, I will plot the percentile position of each observed marginal statistic within each ensemble's distribution. This is the rank-based consistency check of Stedinger and Taylor (1982). A well-behaved ensemble scatters these percentiles around 0.5, while values pinned near 0 or 1 indicate that the ensemble is missing that statistic.

In [ ]:
res_vs = synhydro.verify(vs_ens, Q_monthly, metrics=["marginal"])
res_nowak = synhydro.verify(nowak_ens, Q_monthly, metrics=["marginal"])

sum_vs = res_vs.summary()
sum_vs = sum_vs[(sum_vs["site"] == site) & (sum_vs["kind"] == "scalar")].sort_values("metric")
sum_nowak = res_nowak.summary()
sum_nowak = sum_nowak[(sum_nowak["site"] == site) & (sum_nowak["kind"] == "scalar")].sort_values("metric")

y = np.arange(len(sum_vs))
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.axvspan(0.05, 0.95, color="0.92", zorder=0, label="90 percent band")
ax.axvline(0.5, color="0.5", linestyle="--", linewidth=1, zorder=1)
ax.scatter(sum_vs["obs_percentile"], y + 0.15, s=45, label="Valencia-Schaake")
ax.scatter(sum_nowak["obs_percentile"], y - 0.15, s=45, marker="s", label="Nowak KNN")
ax.set_yticks(y)
ax.set_yticklabels(sum_vs["metric"])
ax.set_xlim(-0.02, 1.02)
ax.set_xlabel("Percentile of observed statistic within the ensemble")
ax.set_title(f"Marginal statistics at {site}")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))
ax.grid(alpha=0.3, axis="x")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig5_verification_percentiles.png", dpi=150, bbox_inches="tight")

The Nowak percentiles cluster between roughly 0.35 and 0.55 for every statistic, which is about as good as this check gets.

Valencia-Schaake is well behaved for the central statistics, with the mean, standard deviation, CV, and flow quantiles all sitting comfortably in the band, but it drifts at the tails. The observed kurtosis falls at the 5th percentile of the ensemble, the skewness falls near the 11th, and the observed minimum falls at the 81st. Taken together, these results show that the Valencia-Schaake ensemble is systematically heavier-tailed than the record, which is the same lognormal-tail behavior we saw in the monthly boxplots.

## Conclusions

I will resist declaring a winner here, because there is not one. There are a few takeaways that I would carry out of this comparison.

- Both methods conserve annual totals exactly and reproduce the mean seasonal cycle. The differences live in within-year variability and the tails.
- The Nowak method buys empirical realism. Monthly distributions, within-year sequencing, and tail behavior are all inherited from the record. The cost is that it can never produce a within-year pattern that has not already happened. With annual input, the donor pool here is just 80 observed years.
- Valencia-Schaake buys a generative model. It synthesizes months outside the observed range, which is exactly what you want for stress tests, and exactly what you should double-check if low flows or extremes drive your results.
- Neither method models the transition between December and the following January. If serial correlation across year boundaries matters for your application, Stedinger and Vogel (1984) is the classical extension.
- When the two methods disagree, that disagreement is itself useful because it measures how sensitive your analysis is to the disaggregation step.

The methodological swap in this post was three lines of code, which is the point of having both methods behind a common interface. If you want to try this yourself, SynHydro installs directly from [GitHub](https://github.com/TrevorJA/SynHydro), this notebook lives at `examples/blog_post/` in the repository, and the [algorithm documentation](https://trevorja.github.io/SynHydro/algorithms/) has the full formulation of each method. As always, feedback and GitHub issues are welcome.

## References

Lall, U., and Sharma, A. (1996). A nearest neighbor bootstrap for resampling hydrologic time series. *Water Resources Research*, 32(3), 679-693. https://doi.org/10.1029/95WR02966

Nowak, K., Prairie, J., Rajagopalan, B., and Lall, U. (2010). A nonparametric stochastic approach for multisite disaggregation of annual to daily streamflow. *Water Resources Research*, 46(8). https://doi.org/10.1029/2009WR008530

Stedinger, J.R., and Taylor, M.R. (1982). Synthetic streamflow generation: 1. Model verification and validation. *Water Resources Research*, 18(4), 909-918. https://doi.org/10.1029/WR018i004p00909

Stedinger, J.R., and Vogel, R.M. (1984). Disaggregation procedures for generating serially correlated flow vectors. *Water Resources Research*, 20(1), 47-56. https://doi.org/10.1029/WR020i001p00047

Valencia, R.D., and Schaake, J.C. (1973). Disaggregation processes in stochastic hydrology. *Water Resources Research*, 9(3), 580-585. https://doi.org/10.1029/WR009i003p00580